---
1. will parse json files from multiple annotators and aggregate the most 
common value. if its count exceeds the majority threshold then we retain it else we set the attribute values as nan

2. also contains code to select only certain rows from the entire dataset. e.g. removing the tag+other images from the dataset
---

In [1]:
import json
%load_ext autoreload
%autoreload 2

## read in all the jsons

In [4]:
import importlib
import womens_shoe_parser
importlib.reload(womens_shoe_parser)

<module 'womens_shoe_parser' from '/home/user/ShoeAnnotationPipeline/scripts/womens_shoe_parser.py'>

In [5]:
from womens_shoe_parser import read_all_annotators_data
all_annotations = read_all_annotators_data(
    annotation_results_folder = "/home/user/label_studio_annotation_results/annotated_women_shoes_n-2_50annotators_set-1",
    prefix_to_exclude_demo = 'annotator_')


  0%|          | 0/51 [00:00<?, ?it/s]

In [4]:
print(
    len(all_annotations),'\n',
    len([ann for ann in all_annotations if len(ann) > 0]),'\n',
# ann
all_annotations[0][0],'\n'
)

50 
 50 
 {'data': '/data/local-files/?d=label-studio/data/women_shoes_n-2/women_shoes_crawled_images_Shoe_Shaft_Style--High_Top_women_shoes--local--Images--274931556094--7.jpg', 'department': 'womens shoes', 'closure': 'lace-up/tie', 'type': 'casual', 'style': 'Sneaker'} 



In [5]:
# %debug
'''
print(
df_annotations.keys(),'\n',#['department'][0]
df_annotations.closure_x,'\n',
df_annotations.closure_y,'\n',
    )
'''
# all_annotations[0]
'''
all_annotations = [ann for ann in all_annotations if len(ann) > 0]
df_annotations1 = [pd.DataFrame.from_dict(ann) for ann in all_annotations]
df_annotations1[2]
'''

# df_annotations = [pd.DataFrame.from_dict(ann) for ann in all_annotations]
# df_annotations1 = outer_merge_annotations(df_annotations)
# df_annotations1.columns

'\nall_annotations = [ann for ann in all_annotations if len(ann) > 0]\ndf_annotations1 = [pd.DataFrame.from_dict(ann) for ann in all_annotations]\ndf_annotations1[2]\n'

## find the most common attribute value across annotations

In [6]:
from womens_shoe_parser import aggregate_annotations,remove_tag_choices
df_annotations = aggregate_annotations(
    all_annotations,
    fields = ['department','closure','type','style'],
)

REMOVE_TAG_CHOICES = True
if REMOVE_TAG_CHOICES:
    df_annotations = remove_tag_choices(df_annotations)
import sys;sys.exit()



annotations merged
#annotations 51021


  0%|          | 0/4 [00:00<?, ?it/s]

department columns combined
department values counted
most common department value chosen
----------------------------------------
closure columns combined
closure values counted
most common closure value chosen
----------------------------------------
type columns combined
type values counted
most common type value chosen
----------------------------------------
style columns combined
style values counted
most common style value chosen
----------------------------------------
department_1 closure_1 type_1 style_1 department_2 closure_2 type_2 style_2 department_3 closure_3 type_3 style_3 department_4 closure_4 type_4 style_4 department_5 closure_5 type_5 style_5 department_6 closure_6 type_6 style_6 department_7 closure_7 type_7 style_7 department_8 closure_8 type_8 style_8 department_9 closure_9 type_9 style_9 department_10 closure_10 type_10 style_10 department_11 closure_11 type_11 style_11 department_12 closure_12 type_12 style_12 department_13 closure_13 type_13 style_13 departme

SystemExit: 

/home/user/miniconda3/envs/annotation-env/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3449: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [23]:
# %debug
'''
k = 'department'
for row in df_annotations[k]:
#     print(row)
    print(max(row,key=row.get))
'''
# df_annotations['department'][~df_annotations['department'].isna()]
# df_annotations
# df_confused

,data,department,closure,type,style,tag-choices
0,/data/local-files/?d=label-studio/data/women_s...,womens shoes,lace-up/tie,casual,Sneaker,NaN
1,/data/local-files/?d=label-studio/data/women_s...,womens shoes,zip,boot,Bootie,NaN
2,/data/local-files/?d=label-studio/data/women_s...,womens shoes,lace-up/tie,athletic,Sneaker,NaN
3,/data/local-files/?d=label-studio/data/women_s...,womens shoes,lace-up/tie,casual,Sneaker,NaN
4,/data/local-files/?d=label-studio/data/women_s...,womens shoes,hook,sandal,NaN,NaN
...,...,...,...,...,...,...
51016,/data/local-files/?d=label-studio/data/women_s...,womens shoes,pull-on/slip-on,flat,NaN,NaN
51017,/data/local-files/?d=label-studio/data/women_s...,other,NaN,NaN,NaN,NaN
51018,/data/local-files/?d=label-studio/data/women_s...,womens shoes,pull-on/slip-on,heel,NaN,NaN
51019,/data/local-files/?d=label-studio/data/women_s...,womens shoes,pull-on/slip-on,boot,Chelsea,NaN


## keep only certain samples:

- copy only shoe images ( discounting tag and other) to a separate folder
- discard all images where the majority label is nan

In [7]:
def discard_non_shoe_images(df):
    df_keep = df.copy()
    df_keep = df_keep.drop(['tag-choices'],axis=1)
    df_only_shoes = df_keep[df['department'] == "womens shoes"]
    df_tags = df_keep[df['department'] == "tag"]
    df_other = df_keep[df['department'] == "other"]
    df_department_nan = df_keep[df['department'].isna()]
    return df_only_shoes,df_tags,df_other,df_department_nan

def discard_confused(df):
    df_keep = df.copy()
    df_keep = df_keep.dropna(how='any',inplace=False)
    df_confused = df[df.isna().any(axis=1)]
    return df_keep,df_confused

def dump_data(df,source_folder,target_folder,N_DUMP):
    for i,fname in enumerate(tqdm.tqdm_notebook(df['data'])):
        fname = fname[len(ls_prefix):]
        source_fname = os.path.join(source_folder,fname)
        target_fname = os.path.join(target_folder,fname)
        assert os.path.exists(source_fname)
        !cp {source_fname} {target_fname}
        if N_DUMP is not None:
            if i > N_DUMP:
                break
def create_folder(folder):
    !rm -rf {folder}
    os.mkdir(folder)


In [8]:
DEBUG = True # will make extra folders for samples we are excluding
             # to visually confirm they are sensible
DUMP = False # =True will make clean folders for dumping selected data
         # !!! DANGEROUS if you already dumped data, will delete that folder

#----------------------------------------------
DISCARD_APART_FROM_SHOE_IMAGES = True # = True if you want to remove (others + tags)
DISCARD_EXAMPLES_WITH_CONFUSION = True # = True if you want to remove confused examples
#----------------------------------------------
N_KEEP_SAMPLES_TO_DUMP = None # None if you want to dump all kept samples, 
                          # a number like 50 if you want a small subset for debugging
N_TAG_SAMPLES_TO_DUMP = 50 # if DEBUG=True, will dump n tag samples in a separate folder
N_OTHER_SAMPLES_TO_DUMP = 50 # if DEBUG=True, will dump n other samples in a separate folder
N_DEPARTMENT_NAN_SAMPLES_TO_DUMP = 50 # if DEBUG=True, will dump n samples with nan in a separate folder
N_CONFUSED_SAMPLES_TO_DUMP = 50 # if DEBUG=True, will dump n confused samples in a separate folder



if DISCARD_APART_FROM_SHOE_IMAGES:
    df_only_shoes,df_tags,df_other,df_department_nan = discard_non_shoe_images(df_annotations)
    df_keep = df_only_shoes

if DISCARD_EXAMPLES_WITH_CONFUSION:
    df_keep,df_confused = discard_confused(df_keep)
else:
    df_confused = pd.DataFrame()

print(f'''shoe images {len(df_keep)}
tag images {len(df_tags)}
other images {len(df_other)}
department nan images (probably confusion in deptt) {len(df_department_nan)}
confused {len(df_confused)}
sum (shoe + tag + other + deptt_nan + confused) {len(df_keep)+len(df_tags)+len(df_other)+len(df_department_nan)+len(df_confused)}
annotations before pruning {len(df_annotations)}
''')

#-----------------------------------------------------------------
ls_prefix = '/data/local-files/?d=label-studio/data/women_shoes_n-2/'
host_containing_dir = '/home/user/women_shoes_n-2/'
folder_for_shoes_no_tags = '/home/user/women_shoes_n-2_no_tags'
folder_for_shoes_tags = '/home/user/women_shoes_n-2_tags'
folder_for_shoes_others = '/home/user/women_shoes_n-2_others'
folder_for_shoes_department_nan = '/home/user/women_shoes_n-2_department_nan'
folder_for_shoes_confused = '/home/user/women_shoes_n-2_confused'


if DUMP:
    import sys;sys.exit() # have put it here to prevent mistakes 
                          # while refactoring, please remove while running
    create_folder(folder_for_shoes_no_tags)
    if DEBUG:
        create_folder(folder_for_shoes_tags)
        create_folder(folder_for_shoes_others)
        create_folder(folder_for_shoes_department_nan)
        create_folder(folder_for_shoes_confused)

    dump_data(df_keep,host_containing_dir,folder_for_shoes_no_tags,N_KEEP_SAMPLES_TO_DUMP)
    print('actual womens shoes dumped')
    if DEBUG:
        dump_data(df_tags,host_containing_dir,folder_for_shoes_tags,N_TAG_SAMPLES_TO_DUMP)
        print('tags dumped')
        dump_data(df_other,host_containing_dir,folder_for_shoes_others,N_OTHER_SAMPLES_TO_DUMP)
        print('other dumped')
        dump_data(df_department_nan,host_containing_dir,folder_for_shoes_department_nan,N_DEPARTMENT_NAN_SAMPLES_TO_DUMP)
        print('department nan dumped')
        dump_data(df_confused,host_containing_dir,folder_for_shoes_confused,N_CONFUSED_SAMPLES_TO_DUMP)
        print('confused samples dumped')


shoe images 20809
tag images 5948
other images 18969
department nan images (probably confusion in deptt) 144
confused 5151
sum (shoe + tag + other + deptt_nan + confused) 51021
annotations before pruning 51021



## gather statistics
see statistics of different attributes

In [9]:
uq_department = df_keep['department'].value_counts()
uq_closure = df_keep['closure'].value_counts()
uq_type = df_keep['type'].value_counts()
uq_style = df_keep['style'].value_counts()
print(uq_department, 
      '\n'+'-'*40 +'\n',
      uq_closure,
      '\n'+'-'*40 +'\n',
      uq_type,
      '\n'+'-'*40 +'\n',
      uq_style,
     )
import sys;sys.exit()


womens shoes    20809
Name: department, dtype: int64 
----------------------------------------
 pull-on/slip-on    8746
lace-up/tie        8225
buckle             2238
zip                 852
hook                319
not_applicable      164
drawstring          151
snap                 96
button               18
Name: closure, dtype: int64 
----------------------------------------
 casual      5576
athletic    5134
heel        4302
boot        2364
sandal      1637
flat        1140
slipper      656
Name: type, dtype: int64 
----------------------------------------
 Sneaker              7426
Pump                 2386
Slip-On              1514
Platform              910
Ballet                883
Bootie                860
Slide                 757
Slingback             682
Gladiator/Strappy     641
Loafer                626
Mary-Jane             469
Dorsay                446
Oxford                432
Clog                  368
Boat                  347
Snow-Boot             335
Western       

SystemExit: 